# Proyek UTS: Klasifikasi Jenis Sampah untuk Daur Ulang

**Dataset:** TrashNet, 6 kelas: `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`.

**Tujuan:** membangun **MLP dan CNN sederhana** sebagai model dasar (*baseline*), lalu membandingkan hasilnya pada data uji yang sama. Notebook disiapkan untuk **Google Colab**.

**Catatan integritas data:** arsip yang diperiksa berisi 2.527 JPEG valid, tetapi 3 pasangan file identik memiliki label berbeda. Notebook ini mengeluarkan **kedua salinan dari setiap pasangan konflik** sebelum membagi data, sehingga tersisa 2.521 gambar. Metadata macOS (`__MACOSX`, `.DS_Store`) juga diabaikan. Catat tindakan ini di laporan.

> Jangan menulis angka akurasi pada laporan sebelum benar-benar menjalankan eksperimen. Simpan tangkapan layar/hasil dari cell evaluasi.

## 1. Impor pustaka dan unggah ZIP

Unggah file `TrashNet, Garbage Classification.zip` dari komputer apabila belum ada di `/content/`. Apabila menggunakan Google Drive, silakan ubah `ZIP_PATH` ke lokasi ZIP di Drive.

In [ ]:
import io, os, zipfile, hashlib, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ZIP_PATH = Path('/content/TrashNet, Garbage Classification.zip')
if not ZIP_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    zip_candidates = [Path('/content') / name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_candidates:
        raise FileNotFoundError('Unggah ZIP dataset TrashNet terlebih dahulu.')
    ZIP_PATH = zip_candidates[0]

print('Dataset ZIP:', ZIP_PATH)
print('TensorFlow:', tf.__version__)

## 2. Periksa dan bersihkan data

Satu file fisik dengan nama mirip gambar di folder `__MACOSX` bukan gambar dan tidak dihitung. Jika hash SHA-256 yang sama muncul pada **kelas yang berbeda**, semua salinannya dikeluarkan agar kita tidak memaksakan label yang saling bertentangan.

In [ ]:
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLEAN_DIR = Path('/content/trashnet_clean')
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
for label in CLASSES:
    (CLEAN_DIR / label).mkdir(exist_ok=True)

records = []
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for member in zf.infolist():
        parts = Path(member.filename).parts
        if (member.is_dir() or len(parts) != 3 or
            parts[0] != 'dataset-resized' or parts[1] not in CLASSES or
            Path(parts[2]).suffix.lower() not in ('.jpg', '.jpeg', '.png')):
            continue
        data = zf.read(member)
        try:
            with Image.open(io.BytesIO(data)) as img:
                img.verify()
        except Exception as exc:
            print('Lewati file tidak valid:', member.filename, exc)
            continue
        records.append({'member': member.filename,
                        'label': parts[1], 'name': parts[2],
                        'sha256': hashlib.sha256(data).hexdigest()})

hash_to_labels = defaultdict(set)
for r in records:
    hash_to_labels[r['sha256']].add(r['label'])
conflicted_hashes = {h for h, labels in hash_to_labels.items() if len(labels) > 1}
conflicts = [r for r in records if r['sha256'] in conflicted_hashes]

print('Gambar valid awal:', len(records))
print('Pasangan/hash dengan label konflik:', len(conflicted_hashes))
print('File yang dikeluarkan:', len(conflicts))
for r in conflicts:
    print('  ', r['member'])

# Buang salinan konflik; jika ada duplikat berlabel sama, simpan satu saja.
kept = []
seen = set()
for r in records:
    if r['sha256'] in conflicted_hashes or r['sha256'] in seen:
        continue
    kept.append(r)
    seen.add(r['sha256'])

with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for r in kept:
        dest = CLEAN_DIR / r['label'] / r['name']
        dest.write_bytes(zf.read(r['member']))

print('Total gambar bersih:', len(kept))
print('Per kelas:', dict(Counter(r['label'] for r in kept)))
assert len(kept) >= 2000, 'Data citra kurang dari persyaratan minimal 2.000 gambar.'
assert len(kept) == 2521, 'Jumlah berbeda dari arsip yang ditinjau; periksa kembali ZIP Anda.'

df = pd.DataFrame([{'filepath': str(CLEAN_DIR / r['label'] / r['name']),
                    'label': r['label']} for r in kept])
display(df.groupby('label').size().rename('Jumlah gambar').to_frame())

## 3. Visualisasi beberapa sampel gambar

In [ ]:
sample = df.groupby('label', group_keys=False).sample(n=1, random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    with Image.open(r['filepath']) as img:
        ax.imshow(img)
    ax.set_title(r['label'])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Bagi train / validation / test (80% / 10% / 10%)

Pembagian **stratified** menjaga proporsi setiap kelas relatif seimbang. Data uji dipisahkan sebelum pelatihan dan tidak digunakan untuk menentukan hyperparameter.

In [ ]:
train_df, tmp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df['label'])
val_df, test_df = train_test_split(
    tmp_df, test_size=0.50, random_state=SEED, stratify=tmp_df['label'])

print('Train:', len(train_df), '| Validation:', len(val_df), '| Test:', len(test_df))
print(pd.DataFrame({'train': train_df['label'].value_counts(),
                    'validation': val_df['label'].value_counts(),
                    'test': test_df['label'].value_counts()}).fillna(0).astype(int))
assert set(train_df.filepath).isdisjoint(val_df.filepath)
assert set(train_df.filepath).isdisjoint(test_df.filepath)
assert set(val_df.filepath).isdisjoint(test_df.filepath)

## 5. Siapkan pipeline TensorFlow

Semua gambar diubah menjadi **64×64 piksel** dan dinormalisasi dari rentang piksel 0–255 menjadi 0–1. Tidak dilakukan augmentasi sehingga dua model dapat dibandingkan secara sederhana.

In [ ]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
AUTO = tf.data.AUTOTUNE
label_to_index = {label: i for i, label in enumerate(CLASSES)}

def decode_image(path, label):
    data = tf.io.read_file(path)
    image = tf.io.decode_jpeg(data, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def build_dataset(dataframe, shuffle=False):
    paths = dataframe['filepath'].astype(str).to_numpy()
    labels = dataframe['label'].map(label_to_index).astype('int32').to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    return ds.map(decode_image, num_parallel_calls=AUTO).batch(BATCH_SIZE).prefetch(AUTO)

train_ds = build_dataset(train_df, shuffle=True)
val_ds = build_dataset(val_df)
test_ds = build_dataset(test_df)
print('Tensor dataset siap.')

## 6. Model baseline pertama: MLP

MLP meratakan gambar menjadi vektor sebelum diproses oleh lapisan Dense. Gunakan `sparse_categorical_crossentropy` karena klasifikasi 6 kelas dengan label integer.

In [ ]:
from tensorflow.keras import layers, Sequential

mlp = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.30),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(CLASSES), activation='softmax')
], name='MLP_baseline')
mlp.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.summary()

## 7. Model baseline kedua: CNN sederhana

CNN memanfaatkan filter konvolusi untuk mengekstraksi pola visual seperti tepi, bentuk, dan tekstur.

In [ ]:
cnn = Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.Conv2D(32, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.30),
    layers.Dense(len(CLASSES), activation='softmax')
], name='CNN_baseline')
cnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

## 8. Pelatihan model

Dua model memakai data, jumlah maksimum epoch, optimizer, dan aturan penghentian awal yang sama. `EarlyStopping` memperbolehkan kedua model berhenti di epoch berbeda jika validasi tidak meningkat.

In [ ]:
EPOCHS = 15

def fit_model(model):
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True)
    return model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS, callbacks=[early_stopping], verbose=1)

history_mlp = fit_model(mlp)
history_cnn = fit_model(cnn)

## 9. Grafik pelatihan

Masukkan kurva **accuracy dan loss** ke laporan. Bandingkan apakah ada selisih performa besar antara train dan validation (*overfitting*).

In [ ]:
OUTPUT_DIR = Path('/content/hasil_uts_trashnet')
OUTPUT_DIR.mkdir(exist_ok=True)

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
for history, name in [(history_mlp, 'MLP'), (history_cnn, 'CNN')]:
    axs[0].plot(history.history['accuracy'], label=f'{name} Train')
    axs[0].plot(history.history['val_accuracy'], linestyle='--', label=f'{name} Validation')
    axs[1].plot(history.history['loss'], label=f'{name} Train')
    axs[1].plot(history.history['val_loss'], linestyle='--', label=f'{name} Validation')
axs[0].set_title('Accuracy selama pelatihan')
axs[1].set_title('Loss selama pelatihan')
for ax in axs:
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160)
plt.show()

## 10. Evaluasi dengan data TEST (jangan digunakan saat training)

Bandingkan hasil MLP dan CNN menggunakan **accuracy**, **loss**, laporan precision/recall/F1, dan confusion matrix. Angka berikut akan muncul setelah notebook dijalankan di Colab, bukan data yang dikarang.

In [ ]:
results = []
for name, model in [('MLP', mlp), ('CNN', cnn)]:
    loss, acc = model.evaluate(test_ds, verbose=0)
    results.append({'Model': name, 'Test loss': loss, 'Test accuracy': acc})
results_df = pd.DataFrame(results)
display(results_df)
results_df.to_csv(OUTPUT_DIR / 'perbandingan_model.csv', index=False)

In [ ]:
y_true = test_df['label'].map(label_to_index).to_numpy()
for name, model in [('MLP', mlp), ('CNN', cnn)]:
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    print(f'\n=== Classification report {name} ===')
    print(classification_report(y_true, y_pred, labels=list(range(len(CLASSES))),
                                target_names=CLASSES, zero_division=0))
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASSES))))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASSES)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=30, colorbar=False)
    ax.set_title(f'Confusion Matrix {name} (TEST)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'confusion_matrix_{name.lower()}.png', dpi=160)
    plt.show()

## 11. Analisis beberapa prediksi salah (CNN)

Hasil ini dapat digunakan untuk pembahasan/keterbatasan model pada laporan UTS.

In [ ]:
cnn_probs = cnn.predict(test_ds, verbose=0)
cnn_pred = np.argmax(cnn_probs, axis=1)
wrong = np.where(cnn_pred != y_true)[0][:5]
if len(wrong) == 0:
    print('Tidak ditemukan prediksi salah pada test set.')
else:
    fig, axes = plt.subplots(1, len(wrong), figsize=(3.4 * len(wrong), 3.5))
    axes = np.atleast_1d(axes)
    for ax, idx in zip(axes, wrong):
        with Image.open(test_df.iloc[idx]['filepath']) as im:
            ax.imshow(im)
        ax.set_title(f'Asli: {CLASSES[y_true[idx]]}\nPrediksi: {CLASSES[cnn_pred[idx]]}')
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'contoh_kesalahan_cnn.png', dpi=160)
    plt.show()

## 12. Simpan model dan catatan eksperimen

In [ ]:
mlp.save(OUTPUT_DIR / 'trashnet_mlp.keras')
cnn.save(OUTPUT_DIR / 'trashnet_cnn.keras')

experiment = {
    'dataset': 'TrashNet', 'jumlah_gambar_asli': len(records),
    'jumlah_gambar_bersih': len(kept),
    'duplikat_konflik_dihapus': len(conflicts),
    'gambar_train': len(train_df), 'gambar_validation': len(val_df),
    'gambar_test': len(test_df), 'img_size': str(IMG_SIZE),
    'batch_size': BATCH_SIZE, 'epoch_maksimal': EPOCHS,
    'random_seed': SEED,
}
with open(OUTPUT_DIR / 'catatan_eksperimen.txt', 'w', encoding='utf-8') as f:
    for k, v in experiment.items():
        f.write(f'{k}: {v}\n')
print('Hasil eksperimen tersimpan di', OUTPUT_DIR)
print('Untuk mengunduh semuanya dari Colab:')
print('!zip -r /content/hasil_uts_trashnet.zip /content/hasil_uts_trashnet')
print('from google.colab import files; files.download("/content/hasil_uts_trashnet.zip")')

## 13. Panduan isi laporan UTS (maksimal 8 halaman)

1. **Pendahuluan:** masalah sampah dan alasan klasifikasi otomatis; rumusan masalah; tujuan.
2. **Tinjauan pustaka:** minimal 3 referensi; MLP, CNN, klasifikasi gambar/TrashNet.
3. **Dataset dan preprocessing:** sumber dan lisensi dataset; 2.527 gambar asli, 6 kelas, pembersihan 6 file berlabel konflik, pembagian data 80/10/10, resize 64×64, normalisasi.
4. **Metodologi:** arsitektur MLP dan CNN (diagram), Adam, learning rate 0,001, batch 32, maksimum 15 epoch, early stopping.
5. **Eksperimen dan hasil:** tabel perbandingan test loss dan test accuracy, grafik training, confusion matrix; isi dengan angka eksperimen nyata.
6. **Pembahasan:** kelas yang sulit, prediksi salah, pengaruh ketidakseimbangan kelas `trash`, keterbatasan dataset.
7. **Kesimpulan dan rencana UAS:** hasil inti dan rencana meningkatkan CNN / transfer learning (contoh MobileNet) serta aplikasi web.
8. **Lampiran:** pembagian peran kelompok, tautan GitHub/GitLab, petunjuk menjalankan notebook.

**Sumber data:** https://github.com/garythung/trashnet. Catat rujukan publikasi dan periksa kewajiban lisensi saat menyusun laporan. Model UTS adalah eksperimen awal; performa pada TrashNet tidak menjamin kemampuan memisahkan sampah dalam kondisi dunia nyata.